In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)

In [0]:
# IMPORTS & CONFIGURATION

from pyspark.sql import functions as F
CATALOG = "retail_demo"
RAW_SCHEMA = "raw"
BASE_PATH = (
    "/Volumes/retail_demo/raw/retail_files/"
    "retail_delta_project"
)

# Incremental source data
INCREMENTAL_PATH = f"{BASE_PATH}/datasets/incremental"
CHECKPOINT_BASE = f"{BASE_PATH}/_checkpoints/phase3"
SCHEMA_BASE = f"{BASE_PATH}/_schemas/phase3"

print(f"Catalog           : {CATALOG}")
print(f"Raw Schema        : {RAW_SCHEMA}")
print(f"Incremental Path  : {INCREMENTAL_PATH}")
print(f"Checkpoint Base   : {CHECKPOINT_BASE}")
print(f"Schema Base       : {SCHEMA_BASE}")

Catalog           : retail_demo
Raw Schema        : raw
Incremental Path  : /Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental
Checkpoint Base   : /Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3
Schema Base       : /Volumes/retail_demo/raw/retail_files/retail_delta_project/_schemas/phase3


In [0]:

# CREATE CHECKPOINT & SCHEMA DIRECTORIES

dbutils.fs.mkdirs(CHECKPOINT_BASE)
dbutils.fs.mkdirs(SCHEMA_BASE)


True

In [0]:
# VERIFY INCREMENTAL SOURCE FILES

print("Incremental source directory:")
print(INCREMENTAL_PATH)
print("\nFiles/directories:")
display(
    dbutils.fs.ls(INCREMENTAL_PATH)
)

Incremental source directory:
/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental

Files/directories:


path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/,day_2026-04-24/,0,1786590611626
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/,day_2026-04-25/,0,1786590611626
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/,day_2026-04-26/,0,1786590611626


In [0]:
# VERIFY ALL INCREMENTAL CSV FILES

days = [
    "day_2026-04-24",
    "day_2026-04-25",
    "day_2026-04-26"
]
all_files = []
for day in days:
    day_path = f"{INCREMENTAL_PATH}/{day}"
    print(f"\n{'=' * 60}")
    print(f"{day}")
    print(f"{'=' * 60}")
    files = dbutils.fs.ls(day_path)
    for file in files:
        print(file.name)
        all_files.append(file.path)

print(f"\nTotal files found: {len(all_files)}")


day_2026-04-24
customers_cdc_2026-04-24.csv
orders_incremental_2026-04-24.csv
products_cdc_2026-04-24.csv

day_2026-04-25
customers_cdc_2026-04-25.csv
orders_incremental_2026-04-25.csv
products_cdc_2026-04-25.csv

day_2026-04-26
customers_cdc_2026-04-26.csv
orders_incremental_2026-04-26.csv
products_cdc_2026-04-26.csv

Total files found: 9


In [0]:
# ORDERS AUTO LOADER STREAM

ORDERS_SCHEMA_PATH = f"{SCHEMA_BASE}/orders"
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", ORDERS_SCHEMA_PATH)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("pathGlobFilter", "orders_incremental_*.csv")
        .option("recursiveFileLookup", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load(INCREMENTAL_PATH)
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("load_type", F.lit("incremental"))
        .withColumn("source_file", F.col("_metadata.file_path"))
)
print("Orders Auto Loader stream created.")
orders_stream.printSchema()

Orders Auto Loader stream created.
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- load_type: string (nullable = false)
 |-- source_file: string (nullable = false)



In [0]:
# WRITE ORDERS TO BRONZE DELTA

ORDERS_CHECKPOINT = f"{CHECKPOINT_BASE}/orders"
ORDERS_TABLE = "retail_demo.raw.bronze_orders_incremental"
orders_query = (
    orders_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", ORDERS_CHECKPOINT)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(ORDERS_TABLE)
)
orders_query.awaitTermination()
print("Orders incremental Bronze load completed.")

Orders incremental Bronze load completed.


In [0]:
# VALIDATE ORDERS BRONZE

display(
    spark.sql("""
        SELECT
            COUNT(*) AS row_count
        FROM retail_demo.raw.bronze_orders_incremental
    """)
)

row_count
6135


In [0]:
# PRODUCTS AUTO LOADER STREAM

from pyspark.sql.functions import current_timestamp, lit, col
products_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option(
            "cloudFiles.schemaLocation",
            f"{SCHEMA_BASE}/products"
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )
        .option(
            "pathGlobFilter",
            "products_*.csv"
        )
        .option("header", "true")
        .option("inferSchema", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load(INCREMENTAL_PATH)
        .withColumn(
            "ingest_ts",
            current_timestamp()
        )
        .withColumn(
            "load_type",
            lit("incremental")
        )
        .withColumn(
            "source_file",
            col("_metadata.file_path")
        )
)
print("Products Auto Loader stream created.")
products_stream.printSchema()

Products Auto Loader stream created.
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- load_type: string (nullable = false)
 |-- source_file: string (nullable = false)



In [0]:
# WRITE PRODUCTS BRONZE

PRODUCTS_BRONZE_CHECKPOINT = f"{CHECKPOINT_BASE}/products_bronze"
products_bronze_query = (
    products_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            PRODUCTS_BRONZE_CHECKPOINT
        )
        .trigger(availableNow=True)
        .toTable(
            "retail_demo.raw.bronze_products_incremental"
        )
)
print("Products Bronze ingestion completed.")

Products Bronze ingestion completed.


In [0]:
# VALIDATE PRODUCTS BRONZE


display(
    spark.sql("""
        SELECT COUNT(*) AS row_count
        FROM retail_demo.raw.bronze_products_incremental
    """)
)

row_count
180


In [0]:
# RESET STORES AUTO LOADER SCHEMA
STORES_SCHEMA_PATH = f"{SCHEMA_BASE}/stores"
print("Deleting old Stores Auto Loader schema:")
print(STORES_SCHEMA_PATH)
dbutils.fs.rm(STORES_SCHEMA_PATH, True)
print("Old Stores schema deleted successfully.")

Deleting old Stores Auto Loader schema:
/Volumes/retail_demo/raw/retail_files/retail_delta_project/_schemas/phase3/stores
Old Stores schema deleted successfully.


In [0]:
# FIND STORE CSV FILES


def find_csv_files(path):
    results = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            results.extend(find_csv_files(item.path))
        elif item.path.lower().endswith(".csv"):
            results.append(item.path)
    return results
all_csv_files = find_csv_files(INCREMENTAL_PATH)
print("CSV files found:")
for f in all_csv_files:
    print(f)
print("\nTotal CSV files:", len(all_csv_files))

CSV files found:
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/customers_cdc_2026-04-24.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/products_cdc_2026-04-24.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/orders_incremental_2026-04-25.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/customers_cdc_2026-04-26.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets

In [0]:
# VERIFY STORES SOURCE

store_files = [
    f
    for f in all_csv_files
    if "store" in f.lower()
]
print("Stores CSV files found:", len(store_files))
if len(store_files) == 0:
    print("No Stores incremental source files found.")
else:
    for f in store_files:
        print(f)

Stores CSV files found: 0
No Stores incremental source files found.


In [0]:
#- INCREMENTAL BRONZE VALIDATION

display(spark.sql("""SELECT load_type,COUNT(*) AS record_count FROM retail_demo.raw.bronze_customers_incremental GROUP BY load_type ORDER BY load_type"""))
display(spark.sql("""SELECT load_type,COUNT(*) AS record_count FROM retail_demo.raw.bronze_products_incremental GROUP BY load_type ORDER BY load_type"""))
display(spark.sql("""SELECT load_type,COUNT(*) AS record_count FROM retail_demo.raw.bronze_orders_incremental GROUP BY load_type ORDER BY load_type"""))

load_type,record_count
incremental,630


load_type,record_count
incremental,180


load_type,record_count
incremental,6135


In [0]:
# ============================================================
# CELL 13C - CHECK CUSTOMER SOURCE FILES
# ============================================================

customer_files = [
    f for f in all_csv_files
    if "customer" in f.lower()
]

print("Customers source files found:", len(customer_files))

for f in customer_files:
    print(f)

Customers source files found: 3
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/customers_cdc_2026-04-24.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/customers_cdc_2026-04-26.csv


In [0]:
#  CUSTOMERS AUTO LOADER STREAM

from pyspark.sql.functions import current_timestamp, lit, col
customers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option(
            "cloudFiles.schemaLocation",
            f"{SCHEMA_BASE}/customers"
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )
        .option("header", "true")
        .option("inferSchema", "true")
        .option(
            "rescuedDataColumn",
            "_rescued_data"
        )
        .option(
            "pathGlobFilter",
            "*customers*.csv"
        )
        .load(INCREMENTAL_PATH)
        .withColumn(
            "ingest_ts",
            current_timestamp()
        )
        .withColumn(
            "load_type",
            lit("incremental")
        )
        .withColumn(
            "source_file",
            col("_metadata.file_path")
        )
)
print("Customers Auto Loader stream created.")

customers_stream.printSchema()

Customers Auto Loader stream created.
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- load_type: string (nullable = false)
 |-- source_file: string (nullable = false)



In [0]:
#  CUSTOMERS BRONZE INGESTION

CUSTOMER_BRONZE_CHECKPOINT = f"{CHECKPOINT_BASE}/customers_bronze_v2"

customers_bronze_query = (
    customers_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CUSTOMER_BRONZE_CHECKPOINT
        )
        .trigger(availableNow=True)
        .toTable(
            "retail_demo.raw.bronze_customers_incremental"
        )
)

print("Customers Bronze ingestion completed.")

Customers Bronze ingestion completed.


In [0]:
# VERIFY CUSTOMERS BRONZE

display(
    spark.sql("""
        SELECT
            COUNT(*) AS row_count
        FROM retail_demo.raw.bronze_customers_incremental
    """)
)


row_count
630


In [0]:
# FINAL PHASE 3 BRONZE VALIDATION

display(
    spark.sql("""
        SELECT
            'customers' AS dataset,
            COUNT(*) AS row_count,
            COUNT(DISTINCT customer_id) AS distinct_ids
        FROM retail_demo.raw.bronze_customers_incremental

        UNION ALL

        SELECT
            'products' AS dataset,
            COUNT(*) AS row_count,
            COUNT(DISTINCT product_id) AS distinct_ids
        FROM retail_demo.raw.bronze_products_incremental

        UNION ALL

        SELECT
            'orders' AS dataset,
            COUNT(*) AS row_count,
            COUNT(DISTINCT order_id) AS distinct_ids
        FROM retail_demo.raw.bronze_orders_incremental
    """)
)

dataset,row_count,distinct_ids
customers,630,582
products,180,166
orders,6135,6000


In [0]:

# LOAD TYPE VALIDATION

display(
    spark.sql("""
        SELECT
            'customers' AS dataset,
            load_type,
            COUNT(*) AS record_count
        FROM retail_demo.raw.bronze_customers_incremental
        GROUP BY load_type

        UNION ALL

        SELECT
            'products' AS dataset,
            load_type,
            COUNT(*) AS record_count
        FROM retail_demo.raw.bronze_products_incremental
        GROUP BY load_type

        UNION ALL

        SELECT
            'orders' AS dataset,
            load_type,
            COUNT(*) AS record_count
        FROM retail_demo.raw.bronze_orders_incremental
        GROUP BY load_type
    """)
)

dataset,load_type,record_count
customers,incremental,630
products,incremental,180
orders,incremental,6135


In [0]:

# CHECKPOINT VALIDATION
print("\nCustomers checkpoint:")
display(dbutils.fs.ls(f"{CHECKPOINT_BASE}/customers_bronze_v2"))

print("\nProducts checkpoint:")
display(dbutils.fs.ls(f"{CHECKPOINT_BASE}/products_bronze"))

print("\nOrders checkpoint:")
display(dbutils.fs.ls(f"{CHECKPOINT_BASE}/orders"))


Customers checkpoint:


path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/customers_bronze_v2/commits/,commits/,0,1786590646633
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/customers_bronze_v2/metadata,metadata,45,1786471848000
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/customers_bronze_v2/offsets/,offsets/,0,1786590646635
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/customers_bronze_v2/sources/,sources/,0,1786590646635



Products checkpoint:


path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/products_bronze/commits/,commits/,0,1786590647533
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/products_bronze/metadata,metadata,45,1786470934000
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/products_bronze/offsets/,offsets/,0,1786590647533
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/products_bronze/sources/,sources/,0,1786590647533



Orders checkpoint:


path,name,size,modificationTime
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/orders/commits/,commits/,0,1786590648279
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/orders/metadata,metadata,45,1786470589000
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/orders/offsets/,offsets/,0,1786590648279
dbfs:/Volumes/retail_demo/raw/retail_files/retail_delta_project/_checkpoints/phase3/orders/sources/,sources/,0,1786590648279
